# Notebook 2 — Dot plot: chrF++ a nivel segmento vs juicio humano

Cruza el **chrF++ por segmento** (calculado en la Notebook 1, sección 1.2) con los
**juicios humanos** de una lingüista con experiencia en qom.

### Los datos de juicio humano

- **28 segmentos por dirección**, tomados del test estratificado de Base.
- Juicios **categóricos** (aproximadamente: correcto / incorrecto / dudoso).
- **No siguen el esquema MQM**: la tipificación MQM es una relectura posterior, así que
  **no hay severidades ni puntajes numéricos**.

> **No** se inventa un escalar de calidad ni se deriva uno a partir de las categorías.

### La pregunta de la figura

¿Los segmentos juzgados **incorrectos** se separan de los **correctos** en el eje de
chrF++, o se mezclan?

## Celda de configuración

Requiere que la **Notebook 1** haya corrido antes (produce `results/chrf_por_segmento.csv`
y `results/traducciones_tidy.csv`). **Completá `JUICIOS_CSV`** (columnas `segment_id,
direction, juicio_humano`) y, si querés, fijá `SISTEMA_EVALUADO`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 20260729
np.random.seed(RANDOM_STATE)

# ── Entradas ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
# CSV de juicios humanos (RELLENAR). Columnas: segment_id, direction, juicio_humano.
JUICIOS_CSV = DATA_DIR / "human_eval/v2-nostrat.csv"

# Salidas de la Notebook 1 (mismos paths).
RESULTS_DIR = Path("data/results")
SEGMENT_CHRF_CSV = RESULTS_DIR / "chrf_por_segmento.csv"
TIDY_CSV         = RESULTS_DIR / "traducciones_tidy.csv"   # para la tabla cualitativa (2.4)

FIG_DIR = Path("poster/figures")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Umbral por debajo del cual no se calculan correlaciones ni tests (n de la categoría).
N_MINIMO_TESTS = 10
N_BOOTSTRAP = 100

# Sistema cuyo chrF++ por segmento se cruza con el juicio humano.
SISTEMA_EVALUADO = "qom-mt-v2-aleatorio"

## 2.1 — Datos: carga, recuperación de `segment_id` y cruce

El CSV de juicios **no trae `segment_id`**, pero sí el texto (`qom` y `es`). Recuperamos el
`segment_id` matcheando ese texto contra `traducciones_tidy.csv` (producido por la Notebook
1), que asocia cada `segment_id` con su `source`/`reference`. El match es sobre texto
**normalizado** (NFC, minúsculas, sin puntuación, espacios colapsados), en cuatro niveles:

1. **par (qom, es)** — el más estricto y preferido;
2. **sólo qom** cuando ese texto identifica un único segmento;
3. **sólo es** ídem;
4. **difuso** (similitud de caracteres ≥ `UMBRAL_DIFUSO`) como último recurso, listado
   aparte para que lo verifiques a mano.

Después cruzamos por `(segment_id, direction)` con el chrF++ por segmento. Se avisa de
categorías inesperadas y de filas sin match; no se completa nada en silencio. Si el CSV ya
trajera `segment_id`, se respeta y no se re-matchea.

In [ ]:
import re
import unicodedata

# ── Normalización de direcciones (misma convención que la Notebook 1) ─────────
DIRECTION_ALIASES = {
    "qom2es": "qom2es", "qom-es": "qom2es", "qom_es": "qom2es", "qomes": "qom2es",
    "qom->es": "qom2es", "qom→es": "qom2es", "tob2spa": "qom2es",
    "es2qom": "es2qom", "es-qom": "es2qom", "es_qom": "es2qom", "esqom": "es2qom",
    "es->qom": "es2qom", "es→qom": "es2qom", "spa2tob": "es2qom",
}
def norm_direction(v):
    k = str(v).strip().lower().replace(" ", "")
    if k in DIRECTION_ALIASES:
        return DIRECTION_ALIASES[k]
    raise ValueError(f"Dirección no reconocida: {v!r}. Agregala a DIRECTION_ALIASES.")

# ── Normalización de texto (para el matching de segment_id) ───────────────────
_PUNT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_ESP  = re.compile(r"\s+")
def normalizar_texto(t):
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = _PUNT.sub(" ", t)
    t = _ESP.sub(" ", t).strip()
    return t

# ── Categorías de juicio esperadas (normalizadas) ─────────────────────────────
CATEGORIAS_ESPERADAS = {"correcto", "incorrecto", "dudoso"}
CAT_ALIASES = {
    "correcto": "correcto", "correcta": "correcto", "ok": "correcto", "bien": "correcto",
    "si": "correcto", "sí": "correcto", "1": "correcto",
    "incorrecto": "incorrecto", "incorrecta": "incorrecto", "mal": "incorrecto",
    "no": "incorrecto", "0": "incorrecto",
    "dudoso": "dudoso", "dudosa": "dudoso", "duda": "dudoso", "?": "dudoso",
    "parcial": "dudoso",
}
def norm_categoria(v):
    return CAT_ALIASES.get(str(v).strip().lower(), str(v).strip().lower())

# ── Carga del CSV de juicios (detecta columnas; segment_id es OPCIONAL) ───────
jr = pd.read_csv(JUICIOS_CSV)
lower = {c.lower().strip(): c for c in jr.columns}
def _col(*cands, req=True):
    for c in cands:
        if c in lower:
            return lower[c]
    if req:
        raise ValueError(f"Falta alguna de estas columnas en {JUICIOS_CSV}: {cands}")
    return None

col_dir    = _col("direction", "dir", "sentido", "direccion", "dirección")
col_juicio = _col("juicio_humano", "juicio", "categoria", "categoría", "label",
                  "evaluacion", "evaluación")
col_qom    = _col("qom", "toba", "tob", "qom_text", req=False)
col_es     = _col("es", "spa", "español", "espanol", "castellano",
                  "reference", "referencia", req=False)
col_seg    = _col("segment_id", "id", "seg_id", req=False)

j = pd.DataFrame({
    "direction":     jr[col_dir].map(norm_direction),
    "juicio_humano": jr[col_juicio].map(norm_categoria),
})
if col_qom is not None: j["qom"] = jr[col_qom].astype("string")
if col_es  is not None: j["es"]  = jr[col_es].astype("string")
if col_seg is not None: j["segment_id"] = jr[col_seg]

inesperadas = set(j["juicio_humano"]) - CATEGORIAS_ESPERADAS
if inesperadas:
    print(f"[aviso] Categorías fuera de {CATEGORIAS_ESPERADAS}: {inesperadas}. "
          "Revisá CAT_ALIASES o el CSV. Se mantienen tal cual.")
print("Filas de juicio:", len(j))
print(j.groupby(["direction", "juicio_humano"]).size().to_string())
print("\nColumnas de texto disponibles:",
      [c for c in ("segment_id", "qom", "es") if c in j.columns])

In [ ]:
# ── Recuperar segment_id por texto contra traducciones_tidy.csv (NB1) ─────────
from difflib import SequenceMatcher

UMBRAL_DIFUSO = 0.90   # similitud mínima para aceptar un match difuso (revisar a mano)

def _indice_segmentos():
    if not Path(TIDY_CSV).exists():
        raise FileNotFoundError(
            f"No existe {TIDY_CSV}: corré primero la Notebook 1 (genera la tabla tidy).")
    tidy = pd.read_csv(TIDY_CSV)
    tidy["direction"] = tidy["direction"].map(norm_direction)
    q2e = (tidy[tidy["direction"] == "qom2es"][["segment_id", "source", "reference"]]
           .rename(columns={"source": "qom", "reference": "es"}))
    e2q = (tidy[tidy["direction"] == "es2qom"][["segment_id", "source", "reference"]]
           .rename(columns={"source": "es", "reference": "qom"}))
    pares = pd.concat([q2e, e2q], ignore_index=True).drop_duplicates("segment_id")
    pares = pares.dropna(subset=["qom", "es"]).reset_index(drop=True)
    pares["qom_norm"] = pares["qom"].map(normalizar_texto)
    pares["es_norm"]  = pares["es"].map(normalizar_texto)
    return pares

def recuperar_segment_ids(j):
    pares = _indice_segmentos()
    by_pair, by_qom, by_es = {}, {}, {}
    for r in pares.itertuples(index=False):
        by_pair.setdefault((r.qom_norm, r.es_norm), r.segment_id)
        by_qom.setdefault(r.qom_norm, set()).add(r.segment_id)
        by_es.setdefault(r.es_norm, set()).add(r.segment_id)
    qom_norms = list(by_qom)   # universo para el fallback difuso

    tiene_qom, tiene_es = "qom" in j.columns, "es" in j.columns
    if not (tiene_qom or tiene_es):
        raise ValueError("El CSV de juicios no trae segment_id ni texto (qom/es): "
                         "no puedo recuperar el id.")

    sids, metodos, scores = [], [], []
    for r in j.itertuples(index=False):
        qn = normalizar_texto(r.qom) if tiene_qom and pd.notna(r.qom) else None
        en = normalizar_texto(r.es)  if tiene_es  and pd.notna(r.es)  else None
        sid, via, sc = None, "sin_match", float("nan")
        if qn and en and (qn, en) in by_pair:
            sid, via, sc = by_pair[(qn, en)], "par_exacto", 1.0
        elif qn and len(by_qom.get(qn, ())) == 1:
            sid, via, sc = next(iter(by_qom[qn])), "solo_qom", 1.0
        elif en and len(by_es.get(en, ())) == 1:
            sid, via, sc = next(iter(by_es[en])), "solo_es", 1.0
        elif qn:   # fallback difuso sobre el lado qom (el más distintivo)
            mejor, mejor_sc = None, 0.0
            for cand in qom_norms:
                s = SequenceMatcher(None, qn, cand).ratio()
                if s > mejor_sc:
                    mejor, mejor_sc = cand, s
            if mejor is not None and mejor_sc >= UMBRAL_DIFUSO and len(by_qom[mejor]) == 1:
                sid, via, sc = next(iter(by_qom[mejor])), "difuso_qom", round(mejor_sc, 3)
        sids.append(sid); metodos.append(via); scores.append(sc)

    out = j.copy()
    out["segment_id"]   = sids
    out["match_metodo"] = metodos
    out["match_score"]  = scores
    return out

# Si el CSV ya trae segment_id completo, se respeta; si no, se recupera por texto.
if "segment_id" in j.columns and j["segment_id"].notna().all():
    print("El CSV ya trae segment_id: se usa tal cual (no se recupera por texto).")
    j["match_metodo"] = "provisto"; j["match_score"] = 1.0
else:
    j = recuperar_segment_ids(j)

# ── Informe del matching ──────────────────────────────────────────────────────
print("Método de recuperación de segment_id:")
print(j["match_metodo"].value_counts().to_string())

difusos = j[j["match_metodo"] == "difuso_qom"]
if len(difusos):
    print(f"\n[revisar] {len(difusos)} match(es) DIFUSO(s) (verificá el texto a mano):")
    cols_d = [c for c in ["direction", "segment_id", "match_score", "qom", "es"]
              if c in difusos.columns]
    print(difusos[cols_d].to_string(index=False))

sin = j[j["segment_id"].isna()]
if len(sin):
    print(f"\n[aviso] {len(sin)} fila(s) SIN segment_id (no matchearon). "
          "Bajá UMBRAL_DIFUSO o revisá el texto:")
    cols_s = [c for c in ["direction", "juicio_humano", "qom", "es"] if c in sin.columns]
    print(sin[cols_s].to_string(index=False))

# Guardamos la versión con ids recuperados (para reusar sin re-matchear).
JUICIOS_CON_ID = RESULTS_DIR / "juicios_con_id.csv"
j.to_csv(JUICIOS_CON_ID, index=False)
print(f"\nJuicios con segment_id -> {JUICIOS_CON_ID}")

# El resto de la notebook usa (segment_id, direction, juicio_humano).
_cols_txt = [c for c in ("qom", "es") if c in j.columns]
j = (j.dropna(subset=["segment_id"])
       [["segment_id", "direction", "juicio_humano"] + _cols_txt]
       .reset_index(drop=True))

In [ ]:
# ── chrF++ por segmento y elección del sistema evaluado ───────────────────────
chrf_seg = pd.read_csv(SEGMENT_CHRF_CSV)
sistemas_disp = sorted(chrf_seg["system"].unique())

if SISTEMA_EVALUADO is None:
    if len(sistemas_disp) == 1:
        sistema = sistemas_disp[0]
        print(f"SISTEMA_EVALUADO no fijado; uso el único presente: {sistema}")
    else:
        raise ValueError(f"Hay varios sistemas en chrf_por_segmento.csv ({sistemas_disp}). "
                         "Fijá SISTEMA_EVALUADO en la config.")
else:
    sistema = SISTEMA_EVALUADO
    if sistema not in sistemas_disp:
        raise ValueError(f"'{sistema}' no está en chrf_por_segmento.csv. "
                         f"Disponibles: {sistemas_disp}")

chrf_sys = chrf_seg[chrf_seg["system"] == sistema][
    ["segment_id", "direction", "chrf_segment"]]

# ── Cruce por (segment_id, direction) ─────────────────────────────────────────
datos = j.merge(chrf_sys, on=["segment_id", "direction"], how="left")
sin_match = datos[datos["chrf_segment"].isna()]
if len(sin_match):
    print(f"[aviso] {len(sin_match)} juicios sin chrF++ (segment_id/direction que no "
          f"matchean con '{sistema}'):")
    print(sin_match[["segment_id", "direction", "juicio_humano"]].to_string(index=False))
datos = datos.dropna(subset=["chrf_segment"]).reset_index(drop=True)
print(f"\nSegmentos cruzados OK: {len(datos)} (sistema: {sistema}).")
datos.to_csv(RESULTS_DIR / "juicio_vs_chrf.csv", index=False)

## 2.2 — Figura principal (dot plot)

Un panel por dirección:

- eje **Y**: chrF++ del segmento;
- eje **X**: categoría de juicio humano, con **jitter** horizontal;
- **color y forma** de marcador según categoría (paleta apta para daltonismo);
- **mediana** de cada grupo con línea horizontal;
- **puntos individuales visibles**: no boxplot ni violín (con este n no corresponde).

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

# Okabe-Ito: color + forma redundantes por categoría (accesibilidad).
ESTILO_CAT = {
    "correcto":   {"color": "#009E73", "marker": "o"},   # verde, círculo
    "dudoso":     {"color": "#E69F00", "marker": "^"},   # naranja, triángulo
    "incorrecto": {"color": "#D55E00", "marker": "s"},   # bermellón, cuadrado
}
ORDEN_CAT = ["incorrecto", "dudoso", "correcto"]

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight", dpi=300)
    print(f"  guardada: {nombre}.pdf y .png (300 dpi)")

rng = np.random.default_rng(RANDOM_STATE)
direcciones = [d for d in ["qom2es", "es2qom"] if d in set(datos["direction"])]

fig, axes = plt.subplots(1, len(direcciones), figsize=(7 * len(direcciones), 7),
                         sharey=True, squeeze=False)
axes = axes[0]
for ax, direccion in zip(axes, direcciones):
    d = datos[datos["direction"] == direccion]
    cats = [c for c in ORDEN_CAT if c in set(d["juicio_humano"])]
    cats += [c for c in sorted(set(d["juicio_humano"])) if c not in cats]
    for x, cat in enumerate(cats):
        g = d[d["juicio_humano"] == cat]
        est = ESTILO_CAT.get(cat, {"color": "#0072B2", "marker": "D"})
        jitter = rng.uniform(-0.18, 0.18, size=len(g))
        ax.scatter(np.full(len(g), x) + jitter, g["chrf_segment"],
                   color=est["color"], marker=est["marker"], s=130,
                   edgecolor="white", linewidth=0.7, alpha=0.9, zorder=3)
        med = g["chrf_segment"].median()
        ax.plot([x - 0.28, x + 0.28], [med, med], color="#222222", lw=3, zorder=4)
        ax.text(x, -6, f"n={len(g)}", ha="center", va="top", fontsize=13, color="#555")
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats)
    ax.set_title(direccion); ax.set_xlabel("juicio humano")
    ax.set_ylim(-8, 105)
axes[0].set_ylabel("chrF++ (segmento)")
fig.suptitle(f"chrF++ por segmento vs juicio humano — {sistema}")
guardar(fig, "figura_dotplot_juicio_chrf")
plt.show()

## 2.3 — Estadística descriptiva, con cautela

- **n por categoría y dirección.** Si alguna categoría tiene **menos de 10 ítems**, no se
  calculan coeficientes de correlación ni tests de hipótesis: se imprime una advertencia
  explícita.
- Sí se reportan **mediana, rango y rango intercuartílico** de chrF++ por categoría.
- Toda medida de asociación que se calcule va con su **IC bootstrap**.

In [ ]:
# ── Descriptivos por dirección x categoría ────────────────────────────────────
filas = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    for cat, g in d.groupby("juicio_humano"):
        x = g["chrf_segment"]
        filas.append({"direction": direccion, "categoria": cat, "n": len(g),
                      "mediana": round(x.median(), 2), "min": round(x.min(), 2),
                      "max": round(x.max(), 2), "q1": round(x.quantile(0.25), 2),
                      "q3": round(x.quantile(0.75), 2),
                      "iqr": round(x.quantile(0.75) - x.quantile(0.25), 2)})
descriptivos = pd.DataFrame(filas).sort_values(["direction", "categoria"])
print(descriptivos.to_string(index=False))
descriptivos.to_csv(RESULTS_DIR / "descriptivos_por_categoria.csv", index=False)

In [ ]:
# ── Asociación juicio-chrF++, sólo si el n lo permite ─────────────────────────
# Codificamos el juicio ordinalmente SÓLO para medir asociación monotónica (Spearman);
# esto no implica inventar un escalar de calidad, es un rango de las 3 categorías.
# Spearman = Pearson sobre rangos; lo implementamos con numpy para no sumar scipy.
ORDEN_ORDINAL = {"incorrecto": 0, "dudoso": 1, "correcto": 2}

def _rankdata(a):
    a = np.asarray(a, dtype=float)
    orden = a.argsort(kind="mergesort")
    r = np.empty(len(a), dtype=float)
    sa = a[orden]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j + 1] == sa[i]:
            j += 1
        r[orden[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r

def spearman(x, y):
    rx, ry = _rankdata(x), _rankdata(y)
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def spearman_ic(x, y, n_boot, rng):
    rho = spearman(x, y)
    idx = np.arange(len(x))
    reps = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        r = spearman(x[s], y[s])
        if not np.isnan(r):
            reps.append(r)
    lo, hi = np.percentile(reps, [2.5, 97.5]) if reps else (np.nan, np.nan)
    return rho, lo, hi

rng_b = np.random.default_rng(RANDOM_STATE + 7)
filas_assoc = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    conteos = d["juicio_humano"].value_counts()
    categorias_ok = all(conteos.get(c, 0) >= N_MINIMO_TESTS for c in ORDEN_ORDINAL)
    usables = d[d["juicio_humano"].isin(ORDEN_ORDINAL)]
    if not categorias_ok:
        chicas = {c: int(conteos.get(c, 0)) for c in ORDEN_ORDINAL}
        print(f"[{direccion}] n por categoría {chicas}: alguna < {N_MINIMO_TESTS}. "
              "NO se calculan correlaciones ni tests de hipótesis "
              "— el tamaño de muestra no lo permite.")
        continue
    x = usables["juicio_humano"].map(ORDEN_ORDINAL).to_numpy()
    y = usables["chrf_segment"].to_numpy()
    rho, lo, hi = spearman_ic(x, y, N_BOOTSTRAP, rng_b)
    filas_assoc.append({"direction": direccion, "spearman_rho": round(rho, 3),
                        "ic_low": round(lo, 3), "ic_high": round(hi, 3),
                        "n": len(usables)})

if filas_assoc:
    assoc = pd.DataFrame(filas_assoc)
    print(assoc.to_string(index=False))
    assoc.to_csv(RESULTS_DIR / "asociacion_spearman.csv", index=False)
else:
    print("No se calcularon medidas de asociación (ver avisos arriba).")

## 2.4 — Casos para inspección cualitativa

Segmentos **juzgados correctos con chrF++ más bajo** y **juzgados incorrectos con chrF++
más alto** (5 de cada uno por dirección), con fuente, referencia e hipótesis. Sirven como
ejemplos para el póster (dónde la métrica y el juicio humano se contradicen).

In [ ]:
# ── Traer fuente/referencia/hipótesis desde la tabla tidy de la Notebook 1 ────
if Path(TIDY_CSV).exists():
    tidy = pd.read_csv(TIDY_CSV)
    tidy["direction"] = tidy["direction"].map(norm_direction)
    tr_txt = tidy[tidy["system"] == sistema][
        ["segment_id", "direction", "source", "reference", "hypothesis"]]
else:
    print(f"[aviso] no existe {TIDY_CSV}: la tabla de casos saldrá sin "
          "fuente/referencia/hipótesis.")
    tr_txt = None

def enriquecer(sub):
    return sub if tr_txt is None else sub.merge(tr_txt, on=["segment_id", "direction"],
                                                how="left")

casos_out = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    correctos = d[d["juicio_humano"] == "correcto"].nsmallest(5, "chrf_segment")
    incorrectos = d[d["juicio_humano"] == "incorrecto"].nlargest(5, "chrf_segment")
    for etiqueta, sub in [("correcto_chrf_bajo", correctos),
                          ("incorrecto_chrf_alto", incorrectos)]:
        s = enriquecer(sub.copy())
        s.insert(0, "caso", etiqueta)
        casos_out.append(s)

if casos_out:
    casos = pd.concat(casos_out, ignore_index=True)
    cols = [c for c in ["caso", "direction", "segment_id", "juicio_humano", "chrf_segment",
                        "source", "reference", "hypothesis"] if c in casos.columns]
    casos = casos[cols]
    print(casos.to_string(index=False))
    casos.to_csv(RESULTS_DIR / "casos_cualitativos.csv", index=False)
    print(f"\nGuardado -> {RESULTS_DIR / 'casos_cualitativos.csv'}")

### Cierre

- La figura contesta de un vistazo si **incorrectos** y **correctos** se separan o se
  mezclan en el eje chrF++.
- Con estos tamaños de muestra, la estadística es **descriptiva**: correlaciones y tests
  sólo si cada categoría llega a `N_MINIMO_TESTS`, siempre con IC bootstrap.
- No se derivó ningún escalar de calidad a partir de los juicios categóricos.